# VIX — notebook complet de réentraînement multi-modèles et stacking

Ce notebook utilise tous les rapports disponibles dans le dossier pour construire un plan d'entraînement, puis réentraîne un maximum de modèles lorsque la base brute de marché est présente.

Il fait trois choses importantes :

1. il lit les fichiers de résultats déjà fournis, y compris les rapports de stacking, walk-forward, EGARCH/SPX, amplitude et résultats itératifs ;
2. il extrait les configurations intéressantes, notamment les horizons, régimes, algorithmes, métriques et features ;
3. il entraîne des modèles de base en walk-forward, génère des prédictions out-of-fold, puis entraîne un meta-modèle de stacking.

Point important : les fichiers de résultats seuls ne contiennent pas forcément toutes les features historiques nécessaires au réentraînement. Si aucune base brute n'est trouvée, le notebook exportera quand même le plan d'entraînement, puis indiquera clairement qu'il faut ajouter un fichier de données.

In [ ]:
from pathlib import Path
import os, re, ast, json, math, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False
try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
    from imblearn.combine import SMOTETomek, SMOTEENN
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMBLEARN = True
except Exception:
    HAS_IMBLEARN = False

BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / 'vix_stacking_outputs'
OUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5
MIN_TRAIN_SIZE = 150
MIN_TEST_SIZE = 30
TOP_N_CONFIGS_PER_GROUP = 10
MAX_FEATURES_PER_MODEL = 100
DEFAULT_HORIZONS = [1, 2, 3, 5, 10, 20]
USE_SAMPLERS = True
USE_OPTIONAL_BOOSTERS = True

REPORT_PATTERNS = ['*report*.xlsx', '*results*.csv', 'VIX_Predictions*.xlsx']
RAW_DATA_CANDIDATES = [
    'vix_dataset.csv','vix_dataset.xlsx','vix_features.csv','vix_features.xlsx',
    'market_data.csv','market_data.xlsx','dataset.csv','dataset.xlsx','data.csv','data.xlsx'
]

print('Dossier courant:', BASE_DIR)
print('Sorties:', OUT_DIR)

In [ ]:
def safe_read_csv(path):
    for enc in ['utf-8', 'utf-8-sig', 'latin1']:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path, sep=';')

def safe_read_excel(path, sheet_name=None):
    return pd.read_excel(path, sheet_name=sheet_name, engine='openpyxl')

def normalize_cols(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def find_col(df, candidates):
    lower = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    for c in df.columns:
        cl = str(c).strip().lower()
        if any(cand.lower() in cl for cand in candidates):
            return c
    return None

def parse_features(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    txt = str(x).strip()
    if txt.lower() in ['', 'nan', 'none', '[]']:
        return []
    try:
        v = ast.literal_eval(txt)
        if isinstance(v, (list, tuple, set)):
            return [str(z).strip() for z in v if str(z).strip()]
    except Exception:
        pass
    txt = txt.replace(';', ',')
    return [p.strip().strip('"\'') for p in txt.split(',') if p.strip()]

def list_inputs():
    rows=[]
    for p in sorted(BASE_DIR.iterdir()):
        if p.is_file():
            rows.append({'file': p.name, 'suffix': p.suffix.lower(), 'size_kb': round(p.stat().st_size/1024,1)})
    return pd.DataFrame(rows)

input_files = list_inputs()
display(input_files)

In [ ]:
def load_reports():
    report_files=set()
    for pat in REPORT_PATTERNS:
        report_files.update(BASE_DIR.glob(pat))
    loaded={}
    for p in sorted(report_files):
        try:
            if p.suffix.lower()=='.xlsx':
                xls=pd.ExcelFile(p, engine='openpyxl')
                loaded[p.name]={}
                for sh in xls.sheet_names:
                    loaded[p.name][sh]=normalize_cols(pd.read_excel(p, sheet_name=sh, engine='openpyxl'))
            elif p.suffix.lower()=='.csv':
                loaded[p.name]={'CSV': normalize_cols(safe_read_csv(p))}
        except Exception as e:
            print('Rapport ignoré:', p.name, e)
    return loaded

reports=load_reports()
print('Rapports chargés:', len(reports))
for f, sheets in reports.items():
    print('-', f, list(sheets.keys()))

In [ ]:
def extract_candidate_configs(reports):
    rows=[]
    metric_candidates=['F1_dir','F1_4cls','F1_macro','F1_weighted','F1','AUC','Accuracy','Acc_dir','Acc_4cls','F1_mean','AUC_mean','Accuracy_mean','Corr','R2']
    for fname, sheets in reports.items():
        for sh, df in sheets.items():
            if df is None or df.empty:
                continue
            feat_col=find_col(df, ['Features','Feature'])
            algo_col=find_col(df, ['Algo','Model','Modèle','Nouveau_Modele','Modele_Reference'])
            h_col=find_col(df, ['Horizon_Days','Horizon','H'])
            reg_col=find_col(df, ['VIX_Regime','Regime','REGIME'])
            train_col=find_col(df, ['Train_Start_Used','Train_Start','Winning_Train_Start'])
            sampler_col=find_col(df, ['Sampler','Best_Sampler','SMOTE_Used'])
            metric_col=find_col(df, metric_candidates)
            if algo_col is None and feat_col is None:
                continue
            for _, r in df.iterrows():
                features=parse_features(r.get(feat_col)) if feat_col else []
                algo=str(r.get(algo_col, 'AUTO')).strip() if algo_col else 'AUTO'
                if algo.lower() in ['nan','none','']:
                    continue
                rows.append({
                    'source_file': fname,
                    'source_sheet': sh,
                    'algo': algo,
                    'horizon': r.get(h_col, np.nan) if h_col else np.nan,
                    'regime': str(r.get(reg_col, 'ALL')).upper().strip() if reg_col else 'ALL',
                    'train_start': r.get(train_col, np.nan) if train_col else np.nan,
                    'sampler': str(r.get(sampler_col, 'none')).strip() if sampler_col else 'none',
                    'features': features,
                    'n_features': len(features),
                    'metric_name': metric_col,
                    'metric_value': pd.to_numeric(r.get(metric_col, np.nan), errors='coerce') if metric_col else np.nan
                })
    cfg=pd.DataFrame(rows)
    if cfg.empty:
        return cfg
    cfg['horizon']=pd.to_numeric(cfg['horizon'], errors='coerce')
    cfg['metric_rank']=cfg['metric_value'].fillna(-999)
    cfg['features_key']=cfg['features'].apply(lambda x: '|'.join(sorted(set(x))))
    cfg=cfg.sort_values(['horizon','regime','metric_rank'], ascending=[True, True, False])
    cfg=cfg.drop_duplicates(['algo','horizon','regime','features_key','sampler'])
    return cfg.drop(columns=['features_key'])

candidate_configs=extract_candidate_configs(reports)
print('Configurations extraites:', len(candidate_configs))
display(candidate_configs.head(30))

cfg_export=OUT_DIR/'candidate_model_configs_from_reports.xlsx'
with pd.ExcelWriter(cfg_export, engine='openpyxl') as writer:
    export=candidate_configs.copy()
    if not export.empty:
        export['features']=export['features'].apply(json.dumps)
    export.to_excel(writer, sheet_name='configs', index=False)
print('Export configs:', cfg_export)

In [ ]:
def load_raw_dataset():
    for name in RAW_DATA_CANDIDATES:
        p=BASE_DIR/name
        if p.exists():
            df=safe_read_csv(p) if p.suffix.lower()=='.csv' else safe_read_excel(p)
            if df is not None and not df.empty:
                print('Base brute détectée:', p.name)
                return normalize_cols(df)
    candidates=[]
    for p in BASE_DIR.iterdir():
        if not p.is_file() or p.suffix.lower() not in ['.csv','.xlsx']:
            continue
        low=p.name.lower()
        if any(k in low for k in ['report','result','prediction','stacking']):
            continue
        try:
            df=safe_read_csv(p) if p.suffix.lower()=='.csv' else safe_read_excel(p)
            candidates.append((p, df.shape[0], df.shape[1], normalize_cols(df)))
        except Exception:
            pass
    if candidates:
        candidates=sorted(candidates, key=lambda z:(z[1],z[2]), reverse=True)
        print('Base brute détectée par fallback:', candidates[0][0].name)
        return candidates[0][3]
    return None

raw_df=load_raw_dataset()
if raw_df is None:
    print('Aucune base brute détectée. Le notebook peut générer le plan, mais pas réentraîner les modèles sans données historiques de features.')
else:
    print(raw_df.shape)
    display(raw_df.head())

In [ ]:
def price_col(df):
    col=find_col(df, ['VIX','VIX_Close','vix_close','Close','Fermeture','Adj Close','PX_LAST','last'])
    if col is not None:
        return col
    nums=[c for c in df.select_dtypes(include=[np.number]).columns if not any(k in c.lower() for k in ['target','pred','prob','signal'])]
    return nums[0] if nums else None

def prepare_dataset(raw_df, horizons):
    df=raw_df.copy()
    dcol=find_col(df, ['Date','Datetime','timestamp'])
    if dcol:
        df[dcol]=pd.to_datetime(df[dcol], errors='coerce')
        df=df.sort_values(dcol).rename(columns={dcol:'Date'}).reset_index(drop=True)
    else:
        df['Date']=pd.RangeIndex(len(df))
    px_col=price_col(df)
    if px_col:
        px=pd.to_numeric(df[px_col], errors='coerce')
        for w in [1,2,3,5,10,20,60,120,252]:
            df[f'vix_ret_{w}d']=px.pct_change(w)
            df[f'vix_ma_{w}d']=px.rolling(w).mean()
            df[f'vix_std_{w}d']=px.rolling(w).std()
            df[f'vix_z_{w}d']=(px-df[f'vix_ma_{w}d'])/df[f'vix_std_{w}d']
        reg=find_col(df, ['VIX_Regime','REGIME','Regime'])
        if reg:
            df['VIX_Regime']=df[reg].astype(str).str.upper().str.strip()
        else:
            q33=px.expanding(min_periods=252).quantile(0.33).shift(1)
            q66=px.expanding(min_periods=252).quantile(0.66).shift(1)
            df['VIX_Regime']=np.select([px<=q33, px>=q66], ['CALM','STRESS'], default='NORMAL')
        for h in horizons:
            fut=px.shift(-int(h))
            delta=fut-px
            df[f'target_dir_h{int(h)}']=(delta>0).astype(int)
            abs_delta=delta.abs()
            threshold=abs_delta.rolling(252, min_periods=50).quantile(0.60).shift(1)
            strength=np.where(abs_delta>=threshold, 'FORT','FAIBLE')
            direction=np.where(delta>0, 'UP','DOWN')
            df[f'target_4cls_h{int(h)}']=pd.Series(direction+'_'+strength, index=df.index)
    else:
        target=find_col(df, ['target','Target','Move','direction','y'])
        if target is None:
            raise ValueError('Aucune colonne VIX/Close/Fermeture ni target détectée.')
        df['VIX_Regime']='ALL'
        for h in horizons:
            y=df[target]
            df[f'target_dir_h{int(h)}']=y.astype(str).str.upper().str.contains('UP|1|TRUE').astype(int) if y.dtype==object else (pd.to_numeric(y, errors='coerce')>0).astype(int)
    return df

if raw_df is not None:
    detected=sorted(candidate_configs['horizon'].dropna().astype(int).unique().tolist()) if not candidate_configs.empty else []
    horizons=sorted(set(DEFAULT_HORIZONS+detected))
    base_df=prepare_dataset(raw_df, horizons)
    print('Horizons:', horizons)
    print(base_df.shape)
    display(base_df.tail())
else:
    horizons=DEFAULT_HORIZONS
    base_df=None

In [ ]:
def model_library(n_classes=2):
    models={
        'LogReg_L2': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
        'RidgeClassifier': RidgeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
        'RandomForest': RandomForestClassifier(n_estimators=400, min_samples_leaf=4, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
        'ExtraTrees': ExtraTreesClassifier(n_estimators=500, min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
        'GradientBoosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
        'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=300, learning_rate=0.035, l2_regularization=0.1, random_state=RANDOM_STATE),
        'AdaBoost': AdaBoostClassifier(n_estimators=250, learning_rate=0.04, random_state=RANDOM_STATE),
        'DecisionTree_balanced': DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, class_weight='balanced', random_state=RANDOM_STATE),
        'KNN_15': KNeighborsClassifier(n_neighbors=15, weights='distance'),
        'SVC_RBF': SVC(C=1.0, gamma='scale', class_weight='balanced', probability=True, random_state=RANDOM_STATE),
        'GaussianNB': GaussianNB(),
        'MLP_small': MLPClassifier(hidden_layer_sizes=(64,32), alpha=0.001, max_iter=700, early_stopping=True, random_state=RANDOM_STATE)
    }
    if USE_OPTIONAL_BOOSTERS and HAS_XGB:
        models['XGBoost']=XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.025, subsample=0.85, colsample_bytree=0.85, objective='binary:logistic' if n_classes==2 else 'multi:softprob', eval_metric='logloss' if n_classes==2 else 'mlogloss', random_state=RANDOM_STATE, n_jobs=-1)
    if USE_OPTIONAL_BOOSTERS and HAS_LGBM:
        models['LightGBM']=LGBMClassifier(n_estimators=600, learning_rate=0.025, num_leaves=31, subsample=0.85, colsample_bytree=0.85, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    if USE_OPTIONAL_BOOSTERS and HAS_CATBOOST:
        models['CatBoost']=CatBoostClassifier(iterations=500, depth=4, learning_rate=0.025, loss_function='Logloss' if n_classes==2 else 'MultiClass', verbose=False, random_state=RANDOM_STATE)
    return models

def sampler_from_name(name):
    if not HAS_IMBLEARN or not USE_SAMPLERS:
        return None
    s=str(name).lower()
    if 'borderline' in s: return BorderlineSMOTE(random_state=RANDOM_STATE, k_neighbors=3)
    if 'adasyn' in s: return ADASYN(random_state=RANDOM_STATE, n_neighbors=3)
    if 'tomek' in s: return SMOTETomek(random_state=RANDOM_STATE)
    if 'enn' in s: return SMOTEENN(random_state=RANDOM_STATE)
    if 'smote' in s or s in ['true','1','yes']: return SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
    return None

def make_pipeline(model, sampler=None, scale=False):
    steps=[('imputer', SimpleImputer(strategy='median'))]
    if scale: steps.append(('scaler', RobustScaler()))
    if sampler is not None and HAS_IMBLEARN:
        return ImbPipeline(steps+[('sampler',sampler),('model',model)])
    return Pipeline(steps+[('model',model)])

def available_features(df, preferred=None):
    forbidden_exact={'Date','Move','REGIME','Regime','VIX_Regime'}
    forbidden=['target','future','pred','prob','signal']
    nums=[c for c in df.select_dtypes(include=[np.number]).columns if c not in forbidden_exact and not any(k in c.lower() for k in forbidden)]
    pref=[f for f in (preferred or []) if f in nums]
    return (pref if pref else nums)[:MAX_FEATURES_PER_MODEL]

def build_training_plan(candidate_configs, df):
    rows=[]
    if candidate_configs is not None and not candidate_configs.empty:
        tmp=candidate_configs.sort_values('metric_value', ascending=False)
        for _, g in tmp.groupby(['horizon','regime'], dropna=False):
            rows.extend(g.head(TOP_N_CONFIGS_PER_GROUP).to_dict('records'))
    regimes=sorted(df['VIX_Regime'].dropna().astype(str).str.upper().unique().tolist()) if df is not None and 'VIX_Regime' in df else ['ALL']
    for h in horizons:
        for reg in ['ALL']+regimes:
            for algo in model_library().keys():
                rows.append({'source_file':'AUTO','source_sheet':'AUTO_GRID','algo':algo,'horizon':h,'regime':reg,'train_start':np.nan,'sampler':'none','features':[],'n_features':0,'metric_name':None,'metric_value':np.nan})
    plan=pd.DataFrame(rows)
    plan['horizon']=pd.to_numeric(plan['horizon'], errors='coerce').fillna(1).astype(int)
    plan['regime']=plan['regime'].astype(str).str.upper().str.strip()
    plan['features_key']=plan['features'].apply(lambda x:'|'.join(sorted(set(x))) if isinstance(x,list) else '')
    plan=plan.drop_duplicates(['algo','horizon','regime','sampler','features_key']).drop(columns='features_key')
    return plan

training_plan=build_training_plan(candidate_configs, base_df) if base_df is not None else None
if training_plan is not None:
    print('Configs à tester:', len(training_plan))
    display(training_plan.head(30))

In [ ]:
def proba_safe(est, X, classes):
    if hasattr(est, 'predict_proba'):
        p=est.predict_proba(X)
        out=np.zeros((len(X), len(classes)))
        est_classes=getattr(est, 'classes_', classes)
        for j,c in enumerate(est_classes):
            if c in classes:
                out[:, list(classes).index(c)]=p[:,j]
        return out
    if hasattr(est, 'decision_function'):
        s=est.decision_function(X)
        if np.ndim(s)==1:
            p1=1/(1+np.exp(-s))
            return np.vstack([1-p1,p1]).T
    pred=est.predict(X)
    out=np.zeros((len(X), len(classes)))
    for i,pred_i in enumerate(pred):
        if pred_i in classes:
            out[i, list(classes).index(pred_i)]=1
    return out

def metrics(y, pred, proba=None):
    out={
        'Accuracy': accuracy_score(y,pred),
        'Precision_macro': precision_score(y,pred,average='macro',zero_division=0),
        'Recall_macro': recall_score(y,pred,average='macro',zero_division=0),
        'F1_macro': f1_score(y,pred,average='macro',zero_division=0),
        'F1_weighted': f1_score(y,pred,average='weighted',zero_division=0)
    }
    try:
        labs=np.unique(y)
        out['AUC']=roc_auc_score(y, proba[:,1]) if proba is not None and len(labs)==2 else (roc_auc_score(y,proba,multi_class='ovr',average='macro') if proba is not None and len(labs)>2 else np.nan)
    except Exception:
        out['AUC']=np.nan
    return out

def train_config(df, cfg, task='dir'):
    h=int(cfg['horizon'])
    target=f'target_dir_h{h}' if task=='dir' else f'target_4cls_h{h}'
    if target not in df:
        return None, None
    work=df.copy()
    reg=str(cfg.get('regime','ALL')).upper()
    if reg!='ALL' and 'VIX_Regime' in work:
        work=work[work['VIX_Regime'].astype(str).str.upper()==reg].copy()
    work=work.dropna(subset=[target]).reset_index(drop=True)
    if len(work)<MIN_TRAIN_SIZE+MIN_TEST_SIZE:
        return None, None
    y_raw=work[target]
    le=None
    if y_raw.dtype==object:
        le=LabelEncoder(); y=le.fit_transform(y_raw.astype(str))
    else:
        y=pd.to_numeric(y_raw, errors='coerce').fillna(0).astype(int).values
    if len(np.unique(y))<2:
        return None, None
    feats=available_features(work, cfg.get('features', []))
    if not feats:
        return None, None
    X=work[feats]
    algo=str(cfg.get('algo','ExtraTrees')).strip()
    lib=model_library(len(np.unique(y)))
    if algo not in lib:
        al=algo.lower()
        algo='LightGBM' if 'light' in al and 'LightGBM' in lib else ('XGBoost' if 'xgb' in al and 'XGBoost' in lib else ('RandomForest' if 'forest' in al else 'ExtraTrees'))
    model=lib[algo]
    scale=algo in ['LogReg_L2','RidgeClassifier','KNN_15','SVC_RBF','GaussianNB','MLP_small']
    sampler=sampler_from_name(cfg.get('sampler','none'))
    classes=np.unique(y)
    oof_pred=np.full(len(work),np.nan)
    oof_proba=np.full((len(work),len(classes)),np.nan)
    for fold,(tr,te) in enumerate(TimeSeriesSplit(n_splits=N_SPLITS).split(X),1):
        if len(tr)<MIN_TRAIN_SIZE or len(te)<MIN_TEST_SIZE:
            continue
        est=make_pipeline(clone(model), sampler=sampler, scale=scale)
        try:
            est.fit(X.iloc[tr], y[tr])
            pred=est.predict(X.iloc[te])
            proba=proba_safe(est, X.iloc[te], classes)
            oof_pred[te]=pred
            oof_proba[te,:]=proba
        except Exception as e:
            print('Erreur modèle:', algo, 'h', h, 'reg', reg, 'fold', fold, e)
    valid=~np.isnan(oof_pred)
    if valid.sum()<MIN_TEST_SIZE:
        return None, None
    pred=oof_pred[valid].astype(int)
    proba=oof_proba[valid]
    met=metrics(y[valid], pred, proba)
    model_id=f'{task}_h{h}_{reg}_{algo}_{abs(hash(str(cfg.to_dict())))%10**8}'
    result={'Model_ID':model_id,'Task':task,'Horizon':h,'Regime':reg,'Algo':algo,'Sampler':cfg.get('sampler','none'),'N_Features':len(feats),'Features':json.dumps(feats),'N_OOF':int(valid.sum()),**met,'Source_File':cfg.get('source_file',''),'Source_Sheet':cfg.get('source_sheet','')}
    pred_df=pd.DataFrame({'Date':work.loc[valid,'Date'].values,'Model_ID':model_id,'Task':task,'Horizon':h,'Regime':reg,'y_true':y[valid],'y_pred':pred})
    for j,c in enumerate(classes):
        pred_df[f'proba_class_{c}']=proba[:,j]
    if le is not None:
        mapping={i:v for i,v in enumerate(le.classes_)}
        pred_df['y_true_label']=pred_df['y_true'].map(mapping)
        pred_df['y_pred_label']=pred_df['y_pred'].map(mapping)
    return result, pred_df

if base_df is not None and training_plan is not None:
    results=[]; preds=[]
    for i,cfg in training_plan.reset_index(drop=True).iterrows():
        for task in ['dir','4cls']:
            r,p=train_config(base_df,cfg,task)
            if r is not None:
                results.append(r); preds.append(p)
                if len(results)%25==0: print('Modèles entraînés valides:', len(results))
    individual_results=pd.DataFrame(results).sort_values(['Task','Horizon','Regime','F1_macro'], ascending=[True,True,True,False]) if results else pd.DataFrame()
    oof_predictions=pd.concat(preds, ignore_index=True) if preds else pd.DataFrame()
    print('Nombre de modèles individuels valides:', len(individual_results))
    display(individual_results.head(30))
else:
    individual_results=pd.DataFrame(); oof_predictions=pd.DataFrame()

In [ ]:
def stack_table(oof, indiv, task, h, top_n=30):
    best=indiv[(indiv.Task==task)&(indiv.Horizon==h)].sort_values('F1_macro', ascending=False).head(top_n)
    if best.empty: return None, None
    ids=best.Model_ID.tolist()
    d=oof[(oof.Task==task)&(oof.Horizon==h)&(oof.Model_ID.isin(ids))].copy()
    if d.empty: return None, None
    prob_cols=[c for c in d.columns if c.startswith('proba_class_')]
    pivots=[]
    for pc in prob_cols:
        p=d.pivot_table(index=['Date','y_true'], columns='Model_ID', values=pc, aggfunc='last')
        p.columns=[f'{m}__{pc}' for m in p.columns]
        pivots.append(p)
    X=pd.concat(pivots, axis=1).sort_index()
    y=X.index.get_level_values('y_true').values
    X=X.reset_index().drop(columns=['Date','y_true'])
    X=X.replace([np.inf,-np.inf], np.nan).dropna(axis=1, how='all')
    return X.fillna(X.median(numeric_only=True)), y

def train_stack(X,y):
    if X is None or len(X)<MIN_TRAIN_SIZE+MIN_TEST_SIZE or len(np.unique(y))<2:
        return None
    candidates={
        'Meta_LogReg':LogisticRegression(max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE),
        'Meta_Ridge':RidgeClassifier(class_weight='balanced',random_state=RANDOM_STATE),
        'Meta_ExtraTrees':ExtraTreesClassifier(n_estimators=500,min_samples_leaf=3,class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1),
        'Meta_RF':RandomForestClassifier(n_estimators=400,min_samples_leaf=4,class_weight='balanced_subsample',random_state=RANDOM_STATE,n_jobs=-1),
        'Meta_HGB':HistGradientBoostingClassifier(max_iter=300,learning_rate=0.03,random_state=RANDOM_STATE)
    }
    if HAS_LGBM:
        candidates['Meta_LightGBM']=LGBMClassifier(n_estimators=400,learning_rate=0.03,class_weight='balanced',random_state=RANDOM_STATE,verbose=-1)
    classes=np.unique(y); rows=[]
    for name,model in candidates.items():
        oof_pred=np.full(len(X),np.nan); oof_proba=np.full((len(X),len(classes)),np.nan)
        for tr,te in TimeSeriesSplit(n_splits=N_SPLITS).split(X):
            if len(tr)<MIN_TRAIN_SIZE or len(te)<MIN_TEST_SIZE: continue
            est=make_pipeline(clone(model), scale=name in ['Meta_LogReg','Meta_Ridge'])
            try:
                est.fit(X.iloc[tr], y[tr])
                oof_pred[te]=est.predict(X.iloc[te])
                oof_proba[te,:]=proba_safe(est, X.iloc[te], classes)
            except Exception as e:
                print('Erreur meta:', name, e)
        valid=~np.isnan(oof_pred)
        if valid.sum()>=MIN_TEST_SIZE:
            rows.append({'Meta_Model':name,'N_OOF':int(valid.sum()),**metrics(y[valid], oof_pred[valid].astype(int), oof_proba[valid])})
    return pd.DataFrame(rows).sort_values('F1_macro', ascending=False) if rows else None

if not individual_results.empty and not oof_predictions.empty:
    stack_rows=[]
    for task in sorted(individual_results.Task.unique()):
        for h in sorted(individual_results.loc[individual_results.Task==task,'Horizon'].unique()):
            X,y=stack_table(oof_predictions, individual_results, task, int(h), top_n=30)
            res=train_stack(X,y)
            if res is not None:
                res.insert(0,'Task',task); res.insert(1,'Horizon',int(h)); res.insert(2,'N_Meta_Features',X.shape[1])
                stack_rows.append(res)
    stacking_results=pd.concat(stack_rows, ignore_index=True) if stack_rows else pd.DataFrame()
    print('Lignes de résultats stacking:', len(stacking_results))
    display(stacking_results.head(30))
else:
    stacking_results=pd.DataFrame()

In [ ]:
report_path=OUT_DIR/'vix_max_stacking_training_report.xlsx'
with pd.ExcelWriter(report_path, engine='openpyxl') as writer:
    c=candidate_configs.copy()
    if not c.empty:
        c['features']=c['features'].apply(json.dumps)
    c.to_excel(writer, sheet_name='Candidate_Configs', index=False)
    if training_plan is not None:
        tp=training_plan.copy(); tp['features']=tp['features'].apply(json.dumps)
        tp.to_excel(writer, sheet_name='Training_Plan', index=False)
    if not individual_results.empty:
        individual_results.to_excel(writer, sheet_name='Individual_Models', index=False)
    if not stacking_results.empty:
        stacking_results.to_excel(writer, sheet_name='Stacking_Results', index=False)
    if not oof_predictions.empty:
        oof_predictions.head(250000).to_excel(writer, sheet_name='OOF_Predictions', index=False)
    pd.DataFrame({
        'Item':['raw_data_loaded','n_candidate_configs','n_training_configs','n_individual_models','n_stacking_rows'],
        'Value':[raw_df is not None, len(candidate_configs), 0 if training_plan is None else len(training_plan), len(individual_results), len(stacking_results)]
    }).to_excel(writer, sheet_name='Summary', index=False)
print('Rapport généré:', report_path)
if raw_df is None:
    print('Action requise: ajoute une base brute comme vix_dataset.csv avec Date + features + VIX ou target, puis relance le notebook.')
else:
    print('Terminé: modèles individuels + stacking générés.')

## Questions utiles si tu veux verrouiller la version suivante

1. La cible finale doit-elle être la direction du VIX uniquement, ou aussi l'amplitude en quatre classes ?
2. Le score principal doit-il être le F1 directionnel, l'AUC, ou un P&L simulé ?
3. Les horizons doivent-ils être tous ceux détectés dans les rapports, ou une liste fixe comme 1, 2, 3, 5, 10 et 20 jours ?
4. Le régime VIX doit-il être repris depuis une colonne existante, ou recalculé automatiquement par quantiles historiques ?
5. Souhaites-tu filtrer les modèles très instables en walk-forward avant de les donner au meta-modèle ?

## Glossaire

- **Stacking** : méthode qui combine plusieurs modèles de base avec un modèle final appelé meta-modèle.
- **Meta-modèle** : modèle entraîné sur les prédictions des autres modèles.
- **OOF, out-of-fold** : prédictions produites sur une période non utilisée pour entraîner le modèle concerné.
- **Walk-forward** : validation temporelle qui respecte l'ordre chronologique des données.
- **F1-score** : mesure qui équilibre la précision et le rappel.
- **AUC** : mesure de la capacité du modèle à classer correctement les hausses et les baisses.
- **Feature** : variable explicative utilisée par un modèle.
- **Régime VIX** : catégorie de marché, par exemple calme, normal ou stressé.